In [1]:
import random
import torch
import torch.nn as nn
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import lovely_tensors as lt

lt.monkey_patch()

In [2]:
base_checkpoint = "HuggingFaceTB/SmolLM2-360M"
checkpoint_path = "checkpoints/hard/checkpoint_epoch_2.pth"
device = "mps" if torch.backends.mps.is_available() else "cpu"
chance_to_remove_end = 0.8
batch_size = 24

In [3]:
class SmolLM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
        self.base_model = AutoModelForCausalLM.from_pretrained(base_checkpoint).to(device)
        self.base_model.lm_head = nn.Identity()
        self.classifier = nn.Sequential(
            # nn.Linear(self.base_model.lm_head.out_features, 1024),
            nn.Linear(960, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        # Freeze smollm2 parameters
        for param in self.base_model.parameters():
            param.requires_grad = False
        # LoRA fine-tuning
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["q_proj", "v_proj", 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
            # Target modules for LoRA
            lora_dropout=0.0,
            bias="none",
            use_dora=True
        )
        self.base_model = get_peft_model(self.base_model, lora_config)
        self.base_model.print_trainable_parameters()
        # self.model.config.output_hidden_states = True

    def forward(self, x):
        # Ensure that the input dictionary contains both "input_ids" and "attention_mask"
        input_ids = x["input_ids"]
        attention_mask = x["attention_mask"]

        # Forward pass through the base model using the attention mask
        out = self.base_model(input_ids, attention_mask=attention_mask)
        logits = out.logits  # shape: (batch_size, seq_len, hidden_dim)

        # Calculate the index of the last non-padding token for each sequence
        last_token_indices = attention_mask.sum(dim=1) - 1  # shape: (batch_size)
        real_batch_size = logits.size(0)
        batch_indices = torch.arange(real_batch_size, device=device)

        # Select logits corresponding to the last non-padding token
        last_logits = logits[batch_indices, last_token_indices, :]  # shape: (batch_size, hidden_dim)

        # Pass the selected logits through the classifier
        output_logits = self.classifier(last_logits)
        return output_logits.squeeze(-1)

In [4]:
checkpoint = torch.load(checkpoint_path, map_location=device)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

model = SmolLM().to(device)

trainable params: 4,618,240 || all params: 366,439,360 || trainable%: 1.2603


In [6]:
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

SmolLM(
  (base_model): PeftModel(
    (base_model): LoraModel(
      (model): LlamaForCausalLM(
        (model): LlamaModel(
          (embed_tokens): Embedding(49152, 960)
          (layers): ModuleList(
            (0-31): 32 x LlamaDecoderLayer(
              (self_attn): LlamaAttention(
                (q_proj): lora.Linear(
                  (base_layer): Linear(in_features=960, out_features=960, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=960, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=8, out_features=960, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict(
                    (defa

In [20]:
with torch.no_grad():
    # Prepare your input data as done during training, e.g., tokenize
    user_input = input("Enter a prompt:")
    print(f"Input: {user_input}")
    inputs = tokenizer(user_input, return_tensors="pt",
                       padding=True, truncation=True).to(device)
    logits = model(inputs)

cutoff_percentage = 0.5    
probs = torch.sigmoid(logits)
predictions = (probs > cutoff_percentage).float()

# print(logits)
print(f"Probability for end of sentence: {probs[0].item()*100}%")
print(f"End of sentence prediction at {int(round(cutoff_percentage,2)*100)}%: {'True' if predictions.item() == 1.0 else 'False'}")

Input: how are you?
Probability for end of sentence: 99.83134269714355%
End of sentence prediction at 50%: True


# Comparison with baseline

In [2]:
base_checkpoint = "HuggingFaceTB/SmolLM2-360M"
checkpoint_path = "checkpoints/hard/checkpoint_epoch_2.pth"
device = "mps" if torch.backends.mps.is_available() else "cpu"
chance_to_remove_end = 0.8
batch_size = 24

tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

In [3]:
class EosDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)
        self.sentence = df["sentence"].tolist()

    def __len__(self):
        return len(self.sentence)

    def __getitem__(self, idx):
        sentence = self.sentence[idx]
        label = 1  # Default label for full sentence
        # Truncate the sentence
        if random.random() < 0.5:
            words = sentence.split()
            if len(words) > 2:  # Ensure at least one word remains
                num_words_to_remove = random.randint(1, len(words) - 2)
                sentence = " ".join(words[:-num_words_to_remove])
                label = 0
        else:
            # Don't truncate the sentence
            random_chance = random.random()
            # delete the symbol at the end of the sentence 80% of the time
            if sentence[-1] in ".!?" and random_chance < chance_to_remove_end:
                sentence = sentence[:-1]

        return {
            'sentence': sentence,
            'eos_label': torch.tensor(label, dtype=torch.float32)
        }

In [4]:
test_dataset = EosDataset("data/test_split.csv")
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Baseline

In [14]:
eos_tokens = ["<|endoftext|>", ".", "?", "!", ";", "\n", "\n\n", "\n\n\n", "\n\n\n\n", ".\"", "\xa0", ".”", ".)",
              ".,", ".\\\\", ".;", "\n ", "\n  ", "\n  ", "\n\n  ", "\n\n   ", "\n\n    "]
eos_ids = set()
for tok in eos_tokens:
    # eos_ids.append(tokenizer.encode(text=tok)[0])
    eos_ids.add(tokenizer.encode(text=tok)[0])
eos_ids = list(eos_ids)

model = AutoModelForCausalLM.from_pretrained(base_checkpoint).to(device)
probabilities_list = []
labels_list = []

for batch in tqdm(test_dataloader, desc="Processing batches"):
    sentences = batch["sentence"]
    labels = batch["eos_label"].to(device)
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Shape: [batch_size, seq_length, vocab_size]

    # Find the actual last token index per sentence
    last_token_indices = attention_mask.sum(dim=1) - 1  # Last actual token index before padding

    # Gather the correct logits for each sentence’s last token
    batch_size_actual = logits.shape[0]
    batch_indices = torch.arange(batch_size_actual, device=device)
    last_token_logits = logits[batch_indices, last_token_indices, :]

    # Compute softmax to get probabilities
    probabilities = F.softmax(last_token_logits, dim=-1)

    # Sum probabilities for all tokens in eos_ids
    eos_probs = probabilities[:, eos_ids].sum(dim=-1)
    probabilities_list.extend(eos_probs.cpu().numpy())
    labels_list.extend(labels.cpu().numpy())

Processing batches: 100%|██████████| 47/47 [00:14<00:00,  3.25it/s]


In [18]:
thresholds = [0.5, 0.7, 0.9]
accuracies = {}

total_eos_samples = labels_list.count(1)
assert total_eos_samples > 0

for threshold in thresholds:
    count_correct_predictions = 0
    for idx, prob in enumerate(probabilities_list):
        if (prob >= threshold and labels_list[idx]==1) or (prob < threshold and labels_list[idx]==0):
            count_correct_predictions += 1
    print(f"Accuracy@{int(threshold * 100)}: {((count_correct_predictions / len(test_dataset)) * 100.0):.2f}%")

Accuracy@50: 73.09%
Accuracy@70: 59.41%
Accuracy@90: 49.29%


# Smoll2 fine-tuning

In [21]:
class SmolLM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
        self.base_model = AutoModelForCausalLM.from_pretrained(base_checkpoint).to(device)
        self.base_model.lm_head = nn.Identity()
        self.classifier = nn.Sequential(
            # nn.Linear(self.base_model.lm_head.out_features, 1024),
            nn.Linear(960, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        # Freeze smollm2 parameters
        for param in self.base_model.parameters():
            param.requires_grad = False
        # LoRA fine-tuning
        lora_config = LoraConfig(
            r=8,
            lora_alpha=32,
            target_modules=["q_proj", "v_proj", 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
            # Target modules for LoRA
            lora_dropout=0.0,
            bias="none",
            use_dora=True
        )
        self.base_model = get_peft_model(self.base_model, lora_config)
        self.base_model.print_trainable_parameters()
        # self.model.config.output_hidden_states = True

    def forward(self, x):
        # out = self.base_model(x["input_ids"])
        # logits = out.logits[:, -1, :]  # Select the last token's logits
        # logits = self.classifier(logits)
        # return logits.squeeze(-1)
        # Ensure that the input dictionary contains both "input_ids" and "attention_mask"
        input_ids = x["input_ids"]
        attention_mask = x["attention_mask"]

        # Forward pass through the base model using the attention mask
        out = self.base_model(input_ids, attention_mask=attention_mask)
        logits = out.logits  # shape: (batch_size, seq_len, hidden_dim)

        # Calculate the index of the last non-padding token for each sequence
        last_token_indices = attention_mask.sum(dim=1) - 1  # shape: (batch_size)
        real_batch_size = logits.size(0)
        batch_indices = torch.arange(real_batch_size, device=device)

        # Select logits corresponding to the last non-padding token
        last_logits = logits[batch_indices, last_token_indices, :]  # shape: (batch_size, hidden_dim)

        # Pass the selected logits through the classifier
        output_logits = self.classifier(last_logits)
        return output_logits.squeeze(-1)

In [27]:
model = SmolLM().to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

probabilities_list = []
labels_list = []

total = len(test_dataset)
all_probs = []
for batch in tqdm(test_dataloader, desc="Processing batches"):
    sentences = batch["sentence"]
    labels = batch["eos_label"].to(device)
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        logits = model(inputs)
    probs = torch.sigmoid(logits)
    all_probs.extend(probs.cpu().numpy())
    labels_list.extend(labels.cpu().numpy())
full_probs = torch.tensor(all_probs, device=device)
labels_tensor = torch.tensor(labels_list, device=device)
    

thresholds = [0.5, 0.7, 0.9]
accuracies = {}
for threshold in thresholds:
    correct = 0
    predictions = (full_probs > threshold).float()
    correct = (predictions == labels_tensor).sum().item()
    print(f"Accuracy@{int(threshold * 100)}: {((correct / total) * 100.0):.2f}%")

trainable params: 4,618,240 || all params: 366,439,360 || trainable%: 1.2603


Processing batches: 100%|██████████| 47/47 [00:28<00:00,  1.66it/s]

Accuracy@50: 95.56%
Accuracy@70: 95.03%
Accuracy@90: 89.88%
